# AI6124 Group 1 — LSTM-Encoded Fuzzy Local Experts for SRT

**Zou Runze (Ryan) · Ngeow Shan Yong Destin**  
**Working Notebook scaffold, 30 October 2026.**

This Notebook starts the final course deliverable. It runs the frozen data audit and a no-change baseline. Trained-model predictions, interpretation, a causal portfolio ledger, and conclusions still have to be added. Protocol v1.2 is in `docs/experiment_protocol_lstm_fuzzy_local_experts.md`; its version-2 export contract is not yet implemented by the legacy evaluator.

**Research question:** Can an LSTM learn three features from recent SRT/SORA/STI data that support a two-rule fuzzy mixture of local experts for next-day raw SRT return prediction?

## Literature review

This section is a pretraining draft. Add the team's measured results and final discussion before submission.

## Method lineage and taxonomy

The proposed model combines two families of methods. An LSTM handles an ordered, multivariate history; a Takagi–Sugeno-style fuzzy layer combines simple local return models. The distinction matters because the LSTM's hidden features are learned from the entire 20-day sequence, while the fuzzy layer receives only a three-dimensional summary at a single forecast time.

1. **Local fuzzy models (1985).** Takagi and Sugeno described fuzzy antecedents whose consequents are local input–output models. This supplies the basic form used here: rule strength times local numerical prediction, followed by aggregation. Our two affine experts are a deliberately small version of that idea, with Gaussian membership functions over learned rather than hand-selected inputs. [Original paper](https://doi.org/10.1109/TSMC.1985.6313399).
2. **Trainable neuro-fuzzy inference (1993).** Jang's ANFIS placed fuzzy inference in an adaptive-network framework and showed how rule parameters could be learned from examples. It is relevant as the classical neuro-fuzzy alternative, but a direct ANFIS on all `20 days × 6 variables` would not express the intended sequential encoder and could make the rule space unnecessarily large. Our LSTM first reduces history to three learned inputs, and all parts are trained jointly by backpropagation. [Original paper](https://doi.org/10.1109/21.256541).
3. **Recurrent sequence representation (1997).** Hochreiter and Schmidhuber introduced LSTM to make learning longer temporal dependencies easier than in conventional recurrent networks. In this project the LSTM is an encoder, and an ordinary LSTM with a scalar return head is the directly relevant benchmark. The benchmark and hybrid receive identical information, so their difference isolates the value of the prediction head more clearly than comparing different datasets or look-back periods. [Original paper](https://doi.org/10.1162/neco.1997.9.8.1735).
4. **Deep sequence encoder plus fuzzy local forecasting (2025).** Ožbot, Škrjanc and Štruc's *Fuzzformer* combines an LSTM, multi-head temporal attention, learned low-dimensional representations, Gaussian fuzzy rules and ARIX local models for multi-horizon S&P 500 forecasting. Their study motivates letting a sequence model learn fuzzy-system inputs. Their paper also uses clustering and rule-separation objectives to support interpretable local models. The present course design removes attention, uses one next-day SRT ETF return, two diagonal-Gaussian rules, two affine experts, and one ordinary MSE objective. It should therefore be described as an **adaptation**, not a reproduction or a claim to obtain the paper's results. [Original paper](https://arxiv.org/html/2510.00960v1).

| Family | Uses time ordering? | Local fuzzy rules? | Role in this project |
| --- | --- | --- | --- |
| No-change return predictor | No learned sequence model | No | Noise-resistant reference for daily ETF returns |
| Ordinary LSTM | Yes | No | Main machine-learning benchmark |
| LSTM → 3D → linear head | Yes | No | Controls for representation compression |
| Classical ANFIS / Takagi–Sugeno-style models | Usually use a fixed-size input vector | Yes | Historical method family; explains rule/consequent form |
| Fuzzformer | Yes, with attention | Yes, with ARIX local models | Recent architecture that motivates the hybrid |
| Our LSTM → 3D → two fuzzy affine experts | Yes | Yes | Small course adaptation on SRT and published local market data |

## Why this adaptation is testable

The first hypothesis is about **forecasting**, not guaranteed superiority: `Fuzzy2` might lower the next-day raw-return RMSE relative to the same-input `LSTM32` or `LSTM3`. The `LSTM3` control is necessary because a better result could come from forcing a compact representation, independently of fuzzy rules. The second hypothesis is about **local specialization**: rule weights and expert outputs might vary meaningfully across observed past-volatility or past-trend conditions. If one rule dominates or both experts predict almost the same return, the intended specialization has not appeared, even if the forecast error is good.

Input-source ablations ask whether adding published SORA and STI improves the SRT forecast beyond SRT history alone. This is a data question, not evidence that either input causes ETF price movements. SORA is aligned by its publication time; the Monetary Authority of Singapore says the rate for a business day is published at 9:00 on the next business day. [MAS publication policy](https://www.mas.gov.sg/monetary-policy/SORA).

The Fuzzformer paper reports experiments on a different market, prediction horizon, target definition, normalization and architecture. Its numeric RMSE values cannot be placed in our result table as a direct head-to-head benchmark. We can compare **method design and observed strengths/limitations** in prose, while our own ordinary LSTM supplies the numerical benchmark on the identical SRT folds.

## Results paragraphs to complete after training

- State the pooled forward-validation and final-holdout return RMSE for `Z0`, `LSTM32`, `LSTM3` and `Fuzzy2`, including seed variation and paired differences on common target dates.
- Explain which data-source additions helped or hurt. If the gain is small compared with seed variation, say so.
- Discuss `K=1/2/3` rule sensitivity, the learned rule-weight distribution and whether expert outputs differ. Do not label a rule “bull” or “crash” based only on its index.
- Compare forecast accuracy with the separately timed, price-only next-open portfolio exercise. State the close-to-close forecast versus open-to-open trading horizon mismatch, cost assumptions and dividend limitation.
- End with limitations and future work: short ETF history, three learned latent features with no direct economic names, possible regime instability, missing one open, and the need to validate distributions before any total-return claim.

## Bibliography for the final Notebook

- Takagi, T., & Sugeno, M. (1985). *Fuzzy identification of systems and its applications to modeling and control*. IEEE Transactions on Systems, Man, and Cybernetics, SMC-15(1), 116–132. https://doi.org/10.1109/TSMC.1985.6313399
- Jang, J.-S. R. (1993). *ANFIS: Adaptive-network-based fuzzy inference system*. IEEE Transactions on Systems, Man, and Cybernetics, 23(3), 665–685. https://doi.org/10.1109/21.256541
- Hochreiter, S., & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation, 9(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- Ožbot, M., Škrjanc, I., & Štruc, V. (2025). *A Neuro-Fuzzy System for Interpretable Long-Term Stock Market Forecasting*. arXiv:2510.00960v1. https://arxiv.org/html/2510.00960v1
- Ang, K. K. (2026). *AI6124 — Week 8: Neural in Neuro-Fuzzy Systems*. Course lecture PDF supplied to the class.

## Data and information timing

The target is the SGD counter of the CSOP iEdge S-REIT Leaders Index ETF (`SRT.SI`). The primary label is the next-trading-day **raw close-to-close log return**. Each forecast uses the latest 20 SRT trading-day feature rows available after the forecast day's close. SORA is aligned by **publication date**. The saved 28 October snapshot combines SRT, STI and MAS SORA; its raw responses and source hashes are retained in `data/`. The 2025-10-24 SRT close was repaired with a documented second source; its open remains unavailable.

We use three expanding forward-time validation folds, followed by an untouched final holdout. No scaler is fitted on validation or test rows. `src/project_eval.py` provides the shared split and scoring contract.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent)
     if (candidate / "data" / "source_manifest.json").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this Notebook from the project root or docs/ directory")
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.project_eval import Dataset, FEATURE_GROUPS, VALIDATION_FOLDS

subprocess.run([sys.executable, "data/verify_dataset.py"], check=True)
data = Dataset()
print("X shape:", data.x.shape)
print("Features:", data.features)

In [ ]:
fold_summary = []
for fold in (*VALIDATION_FOLDS, "final_holdout"):
    role = "test" if fold == "final_holdout" else "validation"
    train_ids, eval_ids = data.ids(fold, "train"), data.ids(fold, role)
    fold_summary.append({
        "fold": fold,
        "train_windows": len(train_ids),
        "eval_windows": len(eval_ids),
        "first_eval_target": data.index.loc[eval_ids[0], "target_date"].date(),
        "last_eval_target": data.index.loc[eval_ids[-1], "target_date"].date(),
    })
display(pd.DataFrame(fold_summary))

In [ ]:
daily = data.daily.loc[data.daily.date <= pd.Timestamp("2025-09-30")].copy()
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].plot(daily.date, daily.srt_close, color="#0c6b76", linewidth=1.2)
axes[0].set_ylabel("SRT close (SGD)")
axes[1].plot(daily.date, daily.sora_rate_pct_asof, color="#d77931", linewidth=1.0)
axes[1].set_ylabel("Published SORA (%)")
axes[2].plot(daily.date, daily.sti_close_asof, color="#52699a", linewidth=1.0)
axes[2].set_ylabel("STI close")
axes[2].set_xlabel("Date")
fig.suptitle("Three source series in the development period")
plt.show()

## Models and benchmark

- `Z0`: predict zero next-day return (unchanged closing price).
- `LSTM32`: one-layer, 32-hidden-unit LSTM with a scalar return head.
- `LSTM3`: the same encoder with a three-dimensional `tanh` projection and scalar linear head.
- `Fuzzy2`: the same encoder and projection, then two diagonal Gaussian rules and two affine return experts.

The fuzzy system receives only the three learned latent values, not a flattened 20 × 6 input. `Fuzzy2` predicts a rule-weighted return. The exact initialization, optimizer, stopping policy, seeds and ablations are fixed in the protocol. Destin will insert model and training cells here. The encoder, fuzzy weights and experts must be trained only on each fold's training rows; the outer validation rows are used only for evaluation.

In [ ]:
# Example of the shared train-only scaling and inner stopping split.
inner = data.prepare("validation_1", "all", inner_stop=True)
outer = data.prepare("validation_1", "all", inner_stop=False)
print("Inner train / stopping:", inner.x_train.shape, inner.x_eval.shape)
print("Outer refit / evaluation:", outer.x_train.shape, outer.x_eval.shape)
print("Outer train scaled target mean:", round(float(outer.y_train.mean()), 6))

## No-change baseline on the three development folds

The no-change predictor needs no fitting and is evaluated only on the development folds here. The final test remains reserved for the frozen set of trained models. Directional accuracy excludes zero true returns; the majority-direction reference is learned from each fold's training labels.

In [ ]:
zero_predictions = data.zero_predictions()
zero_scores = data.forecast_scores(zero_predictions)
zero_pooled = data.pooled_validation_scores(zero_predictions)
display(zero_scores[[
    "fold", "n", "return_rmse", "return_mae", "price_rmse_sgd",
    "direction_n", "zero_actual_n", "train_majority_accuracy"
]])
display(zero_pooled)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(zero_scores.fold, zero_scores.return_rmse, color="#0c6b76")
ax.set_ylabel("Raw next-day log-return RMSE")
ax.set_title("Development-fold no-change baseline")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## Trained-model predictions and result analysis

After the evaluator is migrated to protocol v1.2, Destin's training script should write `output/model_predictions_validation.csv` with one row per outer-validation forecast and these columns:

`model_id,source_group,rule_count,fold,seed,window_id,y_pred`

`y_pred` must be in **raw log-return units**, after inverse target scaling. The current evaluator still groups by the old six-column key and must be upgraded before it scores mixed source/rule configurations. Fuzzy case and rule-long diagnostics use the full key; see the protocol. Do not infer that a rule is a particular market regime from its numeric label.

In [ ]:
prediction_file = Path("output/model_predictions_validation.csv")
model_predictions = None
if prediction_file.exists():
    candidate = pd.read_csv(prediction_file)
    if "rule_count" in candidate.columns:
        print("Version-2 predictions found. Add the upgraded evaluator before scoring them.")
    else:
        model_predictions = candidate  # Legacy single-configuration files only.
        model_scores = data.forecast_scores(model_predictions)
        model_pooled = data.pooled_validation_scores(model_predictions)
        display(model_scores.sort_values(["model_id", "source_group", "fold", "seed"]))
        display(model_pooled.sort_values("return_rmse"))
        if {"Fuzzy2", "LSTM32"} <= set(model_predictions.model_id):
            display(data.paired_validation_rmse_difference(model_predictions))
else:
    print(f"Awaiting trained-model predictions: {prediction_file}")

In [ ]:
if model_predictions is not None:
    joined = data.validate_predictions(model_predictions)
    primary = joined.loc[
        (joined.model_id == "Fuzzy2") & (joined.source_group == "all")
        & (joined.seed == 11) & (joined.fold != "final_holdout")
    ].sort_values("target_date")
    if len(primary):
        fig, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
        axes[0].plot(primary.target_date, primary.y_true, label="Actual", linewidth=1)
        axes[0].plot(primary.target_date, primary.y_pred, label="Fuzzy2", linewidth=1)
        axes[0].set_ylabel("Next-day log return")
        axes[0].legend()
        overview = model_scores.loc[model_scores.fold != "final_holdout"].groupby(
            ["model_id", "source_group"], as_index=False
        ).return_rmse.mean().sort_values("return_rmse")
        labels = overview.model_id + " / " + overview.source_group
        axes[1].barh(labels, overview.return_rmse, color="#0c6b76")
        axes[1].set_xlabel("Mean fold return RMSE (descriptive; use pooled scores for ranking)")
        plt.show()
    else:
        print("Awaiting full-feature Fuzzy2, seed 11, for the forecast plot.")

## Fuzzy-rule diagnostics

Protocol v1.2 requires a case table with the full `(model_id,source_group,rule_count,fold,seed,window_id)` key and `z1,z2,z3,y_mix_raw`, plus a rule-long table with that key and `rule_id,alpha,expert_raw,log_membership,distance2`. Exactly `rule_count` rule rows must match each prediction and their weighted raw expert outputs must equal `y_pred`. The validator and analysis cells remain to implement; rule-number switching across seeds must be considered before attaching market labels.

In [ ]:
case_file = Path("output/fuzzy_cases_validation.csv")
rule_file = Path("output/fuzzy_rules_validation.csv")
if case_file.exists() or rule_file.exists():
    print("Version-2 fuzzy diagnostics found. Add the full-key validator and local-error analysis before display.")
else:
    print(f"Awaiting fuzzy diagnostics: {case_file} and {rule_file}")

## Portfolio application — exploratory price-only evaluation

A signal formed after close `t` is first executable at the next SRT open. The realized open-to-open interval differs from the close-to-close forecast label. Protocol v1.2 requires a causal open-by-open wealth ledger, a model-free momentum control, terminal compounded wealth and an audited dividend treatment. The legacy `portfolio_scores` uses future quote validity to decide an earlier exit at the 2025-10-24 missing open; its output is **not** an executable full-path result. This cell remains a placeholder until that code is replaced and tested.

In [ ]:
print("Causal wealth comparison pending: implement and test the protocol v1.2 ledger before use.")

## Discussion and conclusion — complete after training

1. **Forecast comparison.** Report each development fold, three seeds, pooled validation scores and the once-opened final holdout. Explain the size and variability of `Fuzzy2 − LSTM32` and `Fuzzy2 − LSTM3` differences.
2. **Sources and parameters.** Explain SRT-only versus +SORA versus +STI, and the one/two/three-rule diagnostic. Describe when additional inputs or rules fail to help.
3. **Interpretability.** Plot both rule weights and expert predictions; check their distribution and relationship with *past-only* volatility and momentum. State if rules collapse or swap labels across seeds.
4. **Trading.** Compare price-only next-open performance at three costs with always-long/cash, and acknowledge the forecast/trading-horizon mismatch and unverified distributions.
5. **Limitations and future research.** Address short ETF history, possible overfitting, learned latent features without fixed economic meaning, one missing open, distribution verification, and possible future tests on a longer REIT series.

This section is deliberately left without numerical conclusions until all results have been produced.